### 07 - Evaluate CNN Classifier

##### 1. Objective

Evaluate the trained custom CNN on the untouched test dataset.

This notebook will:

1. Recreate the image DataLoaders from the persistent dataset.
2. Load the trained CNN from MLflow.
3. Generate predictions for the complete test dataset.
4. Calculate:
   - Accuracy
   - Precision
   - Recall
   - F1-score
   - Confusion matrix
5. Analyze class-specific performance.
6. Identify missed defects and false defect alarms.
7. Establish evaluation results for later Explainable AI and
   Responsible AI analysis.

The test dataset is used only for final evaluation and is not used
for training or model selection.

##### 2. Imports

In [0]:
import mlflow.pytorch
import numpy as np
import torch

import matplotlib.pyplot as plt

from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
)

from src.data_preparation import (
    create_image_dataloaders,
)

from src.model_evaluation import (
    evaluate_classifier,
)

from src.project_config import (
    CUSTOM_NORMALIZE_MEAN,
    CUSTOM_NORMALIZE_STD,
)

Notice that we are not importing:

ProductDefectCNN
train_model
Adam
CrossEntropyLoss

because we're not training.

MLflow will reconstruct the trained model.

##### 3. Recreate the DataLoaders

In [0]:
data = create_image_dataloaders()

print("Classes:", data.class_names)
print("Class mapping:", data.class_to_idx)
print("Number of classes:", data.num_classes)

print("Train images:", len(data.train_dataset))
print("Validation images:", len(data.val_dataset))
print("Test images:", len(data.test_dataset))

Remember what is happening here:

Existing images in Volume
          ↓
create_image_dataloaders()
          ↓
PyTorch Dataset/DataLoader objects
created in this new Python session

No images are copied and nothing is trained.

##### 4. Load the persisted model

In [0]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

CNN_RUN_ID = (
    "8b154a58aaae400fbb941a7d8299dc19"
)

MODEL_URI = (
    f"runs:/{CNN_RUN_ID}/model"
)

model = mlflow.pytorch.load_model(
    MODEL_URI
)

model = model.to(device)

print("Evaluation device:", device)
print("CNN loaded successfully.")

In [0]:
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print(
    "Total model parameters:",
    f"{total_parameters:,}",
)

##### 5. Evaluate the complete test set

In [0]:
result = evaluate_classifier(
    model=model,
    data_loader=data.test_loader,
    device=device,
    num_classes=data.num_classes,
)

In [0]:
print(
    "Labels shape:",
    result.labels.shape,
)

print(
    "Predictions shape:",
    result.predictions.shape,
)

print(
    "Probabilities shape:",
    result.probabilities.shape,
)

print(
    f"Test Accuracy: "
    f"{result.accuracy:.4f}"
)

##### 6. Per-class metrics

In [0]:
for class_index, class_name in enumerate(
    data.class_names
):

    print(
        f"\nClass: {class_name}"
    )

    print(
        f"Support: "
        f"{result.support[class_index]}"
    )

    print(
        f"Precision: "
        f"{result.precision[class_index]:.4f}"
    )

    print(
        f"Recall: "
        f"{result.recall[class_index]:.4f}"
    )

    print(
        f"F1-score: "
        f"{result.f1[class_index]:.4f}"
    )

##### 7. Classification report

In [0]:
report = classification_report(
    result.labels,
    result.predictions,
    target_names=data.class_names,
    digits=4,
    zero_division=0,
)

print(report)

##### 8. Confusion matrix

In [0]:
cm = result.confusion_matrix

print(cm)

In [0]:
defect_index = data.class_to_idx[
    "def_front"
]

ok_index = data.class_to_idx[
    "ok_front"
]

correct_defects = cm[
    defect_index,
    defect_index,
]

missed_defects = cm[
    defect_index,
    ok_index,
]

false_defect_alarms = cm[
    ok_index,
    defect_index,
]

correct_ok = cm[
    ok_index,
    ok_index,
]

print(
    "Correct defect detections:",
    correct_defects,
)

print(
    "Missed defects:",
    missed_defects,
)

print(
    "False defect alarms:",
    false_defect_alarms,
)

print(
    "Correct OK predictions:",
    correct_ok,
)

##### 9. Explicit defect metrics

In [0]:
defect_recall = result.recall[
    defect_index
]

missed_defect_rate = (
    1.0 - defect_recall
)

print(
    f"Defect Recall: "
    f"{defect_recall:.4f}"
)

print(
    f"Missed Defect Rate: "
    f"{missed_defect_rate:.4f}"
)

##### 10. Confusion matrix visualization

In [0]:
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=data.class_names,
)

display.plot(
    values_format="d",
)

plt.title(
    "Custom CNN - Test Confusion Matrix"
)

plt.show()

##### 11. Identify failure cases

In [0]:
#All incorrect predictions

incorrect_indices = np.where(
    result.labels
    != result.predictions
)[0]

print(
    "Incorrect predictions:",
    len(incorrect_indices),
)

In [0]:
#Missed defects

missed_defect_indices = np.where(
    (result.labels == defect_index)
    &
    (result.predictions == ok_index)
)[0]

print(
    "Missed defects:",
    len(missed_defect_indices),
)

##### 12. High-confidence errors

In [0]:
incorrect_confidences = np.array([
    result.probabilities[
        index,
        result.predictions[index],
    ]
    for index in incorrect_indices
])

sorted_positions = np.argsort(
    incorrect_confidences
)[::-1]

high_confidence_error_indices = (
    incorrect_indices[
        sorted_positions
    ]
)

In [0]:
for test_index in (
    high_confidence_error_indices[:10]
):

    actual_index = result.labels[
        test_index
    ]

    predicted_index = result.predictions[
        test_index
    ]

    confidence = result.probabilities[
        test_index,
        predicted_index,
    ]

    print(
        f"Index: {test_index:3d} | "
        f"Actual: "
        f"{data.class_names[actual_index]:10s} | "
        f"Predicted: "
        f"{data.class_names[predicted_index]:10s} | "
        f"Confidence: {confidence:.4f}"
    )

##### 13. One improvement to the visualization helper

In [0]:
def denormalize_image(
    image_tensor,
):
    mean = torch.tensor(
        CUSTOM_NORMALIZE_MEAN,
        dtype=image_tensor.dtype,
    ).view(3, 1, 1)

    std = torch.tensor(
        CUSTOM_NORMALIZE_STD,
        dtype=image_tensor.dtype,
    ).view(3, 1, 1)

    image = (
        image_tensor.cpu() * std + mean
    )

    image = image.permute(
        1,
        2,
        0,
    ).numpy()

    return np.clip(
        image,
        0.0,
        1.0,
    )

In [0]:
test_index = incorrect_indices[0]

image_tensor, actual_index = (
    data.test_dataset[test_index]
)

predicted_index = result.predictions[
    test_index
]

confidence = result.probabilities[
    test_index,
    predicted_index,
]

image = denormalize_image(
    image_tensor
)

plt.figure(figsize=(4, 4))

plt.imshow(image)

plt.title(
    f"Actual: "
    f"{data.class_names[actual_index]}\n"
    f"Predicted: "
    f"{data.class_names[predicted_index]}\n"
    f"Confidence: {confidence:.4f}"
)

plt.axis("off")
plt.show()

##### 14. ⚖️ Responsible AI — why missed defects matter

Overall accuracy alone is insufficient for a product-defect
inspection system.

A particularly important failure mode is:

- Actual class = `def_front`
- Predicted class = `ok_front`

This represents a defective casting being classified as acceptable.

Therefore, evaluation should include class-specific recall,
precision, F1-score, and the confusion matrix rather than relying
only on overall accuracy.

The acceptable error rate depends on the real operational context,
cost of errors, safety requirements, and human-review process.

This is a genuine ⚖️ Responsible AI concept.

We're not simply asking: Is accuracy high?

We're asking: What kinds of mistakes does the system make, and what could those mistakes mean in the intended application?

##### 15. 🔍 Explainable AI connection


The confusion matrix identifies which predictions are incorrect,
but it does not explain why the CNN made those predictions.

Later, Grad-CAM will be used to inspect which image regions
influenced selected predictions, including:

- correct defect detections,
- missed defects,
- false defect alarms,
- correct OK classifications.

This connects model evaluation with Explainable AI.

Evaluation
"What did the model get wrong?"

              ↓

🔍 XAI
"Why might the model have made
that prediction?"

Model confidence is not the same thing as certainty or correctness.

##### 07_evaluate_cnn_classifier architecture

``` text

1. Objective
       ↓
2. Imports
       ↓
3. Recreate DataLoaders
       ↓
4. Load persisted CNN from MLflow
       ↓
5. Evaluate test dataset
       ↓
6. Overall accuracy
       ↓
7. Precision / Recall / F1 / Support
       ↓
8. Classification report
       ↓
9. Confusion matrix
       ↓
10. Analyze missed defects
       ↓
11. Inspect incorrect predictions
       ↓
12. Inspect high-confidence errors
       ↓
13. 🔍 XAI observations
       ↓
14. ⚖️ RAI observations
       ↓
15. Key learnings / conclusion

```


``` text

Persistent storage
─────────────────────────────────

Unity Catalog Volume
    └── images

MLflow
    └── trained CNN


Reusable source code
─────────────────────────────────

src/data_preparation.py
    └── DataLoaders

src/model_evaluation.py
    └── evaluation logic


Notebook 07
─────────────────────────────────

orchestration
+
visualization
+
interpretation

```

##### Notebook Conclusion

The custom CNN was evaluated on the untouched test dataset containing
715 casting images.

##### Test Results

- Test images: 715
- Correct predictions: 595
- Incorrect predictions: 120
- Test accuracy: 83.22%

##### Confusion Matrix

| Actual | Predicted def_front | Predicted ok_front |
|---|---:|---:|
| def_front | 385 | 68 |
| ok_front | 52 | 210 |

The model correctly identified 385 defective images and 210 acceptable
images.

However, 68 images labeled `def_front` were incorrectly classified as
`ok_front`. These represent missed defects within the dataset.

The model also classified 52 `ok_front` images as `def_front`, producing
false defect alarms.

Defect recall was approximately 84.99%, corresponding to a missed-defect
rate of approximately 15.01%.

##### 🔍 Explainable AI Observation

The evaluation identifies which predictions are correct or incorrect,
but it does not explain why the CNN made those predictions.

Later, Grad-CAM will be used to investigate which image regions
influenced selected CNN predictions, including correct predictions,
missed defects, false defect alarms, and high-confidence errors.

Grad-CAM indicates regions that influenced the model's prediction. It
does not by itself establish the true physical location of a defect.

##### ⚖️ Responsible AI Observation

Overall accuracy alone is insufficient for evaluating a defect
classification system.

Class-specific precision, recall, F1-score, missed defects, false alarms,
confidence, and operational consequences should also be considered.

The dataset provides image-level `def_front` and `ok_front` labels but
does not provide ground-truth defect locations. Therefore, this project
can objectively evaluate image classification performance but cannot
objectively evaluate defect localization.

##### Key Learnings

1. MLflow allows the trained CNN to be loaded and evaluated without
   retraining.
2. The held-out test dataset provides an independent estimate of model
   classification performance.
3. Accuracy measures overall correctness.
4. Precision measures how reliable predictions of a particular class are.
5. Recall measures how many actual examples of a class are identified.
6. F1-score balances precision and recall.
7. Support is the number of actual examples belonging to each class.
8. The confusion matrix exposes specific types of classification errors.
9. Model confidence does not guarantee correctness.
10. Image classification identifies the class of an image but does not
    inherently locate the physical defect.